In [2]:
import json
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round as spark_round

# Inisialisasi Koneksi MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
collection = db["menu"]

# Bersihkan collection lama jika ada (untuk pengujian ulang yang bersih)
collection.drop()

In [4]:
data_menu = [
    # Kategori Kopi (Wajib: nama_menu, harga, kategori, tingkat_kepahitan)
    {"nama_menu": "Espresso Single", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 20},
    {"nama_menu": "Kopi Tubruk Gayo", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 4, "rekomendasi_barista": True},
    {"nama_menu": "Americano Toraja", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 15},
    {"nama_menu": "Cafe Latte", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 2, "rekomendasi_barista": True},
    {"nama_menu": "Cappuccino", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 18},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 2, "rekomendasi_barista": True},

    # Kategori Makanan (Wajib: nama_menu, harga, kategori, pedas_level)
    {"nama_menu": "Nasi Goreng Rempah", "harga": 32000, "kategori": "Makanan", "pedas_level": 2, "stok": 10},
    {"nama_menu": "Mie Goreng Jawa", "harga": 25000, "kategori": "Makanan", "pedas_level": 1, "stok": 12},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 18000, "kategori": "Makanan", "pedas_level": 0, "rekomendasi_barista": False},
    {"nama_menu": "Cireng Bumbu Rujak", "harga": 14000, "kategori": "Makanan", "pedas_level": 2, "stok": 25},
    {"nama_menu": "Ayam Geprek Sambal Korek", "harga": 28000, "kategori": "Makanan", "pedas_level": 3, "stok": 8},

    # Kategori Non-Kopi / Tambahan Fleksibel
    {"nama_menu": "Matcha Latte", "harga": 26000, "kategori": "Non-Kopi", "manis_level": 3, "stok": 15}
]

# Eksekusi Penyisipan Dokumen
insert_result = collection.insert_many(data_menu)
print(f"Berhasil menyisipkan {len(insert_result.inserted_ids)} dokumen ke collection 'menu'.\n")

Berhasil menyisipkan 12 dokumen ke collection 'menu'.



In [6]:
print("Kopi dengan tingkat_kepahitan > 3:")
kueri_1 = collection.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}})
for doc in kueri_1:
    print(f" - {doc['nama_menu']} (Kepahitan: {doc['tingkat_kepahitan']}, Harga: Rp{doc['harga']})")

print("\nMenu dengan harga antara 15.000 dan 30.000:")
kueri_2 = collection.find({"harga": {"$gte": 15000, "$lte": 30000}})
for doc in kueri_2:
    print(f" - {doc['nama_menu']} | Kategori: {doc['kategori']} | Harga: Rp{doc['harga']}")

print("\nMenu yang TIDAK memiliki field 'pedas_level':")
kueri_3 = collection.find({"pedas_level": {"$exists": False}})
for doc in kueri_3:
    print(f" - {doc['nama_menu']} ({doc['kategori']})")

Kopi dengan tingkat_kepahitan > 3:
 - Espresso Single (Kepahitan: 5, Harga: Rp18000)
 - Kopi Tubruk Gayo (Kepahitan: 4, Harga: Rp15000)
 - Americano Toraja (Kepahitan: 4, Harga: Rp22000)

Menu dengan harga antara 15.000 dan 30.000:
 - Espresso Single | Kategori: Kopi | Harga: Rp18000
 - Kopi Tubruk Gayo | Kategori: Kopi | Harga: Rp15000
 - Americano Toraja | Kategori: Kopi | Harga: Rp22000
 - Cafe Latte | Kategori: Kopi | Harga: Rp28000
 - Cappuccino | Kategori: Kopi | Harga: Rp27000
 - Kopi Susu Gula Aren | Kategori: Kopi | Harga: Rp20000
 - Mie Goreng Jawa | Kategori: Makanan | Harga: Rp25000
 - Roti Bakar Cokelat | Kategori: Makanan | Harga: Rp18000
 - Ayam Geprek Sambal Korek | Kategori: Makanan | Harga: Rp28000
 - Matcha Latte | Kategori: Non-Kopi | Harga: Rp26000

Menu yang TIDAK memiliki field 'pedas_level':
 - Espresso Single (Kopi)
 - Kopi Tubruk Gayo (Kopi)
 - Americano Toraja (Kopi)
 - Cafe Latte (Kopi)
 - Cappuccino (Kopi)
 - Kopi Susu Gula Aren (Kopi)
 - Matcha Latte (Non-

In [7]:
# 1. Menambahkan field 'status' = 'Tersedia' ke seluruh dokumen
update_all_result = collection.update_many({}, {"$set": {"status": "Tersedia"}})
print(f"update_many() berhasil memperbarui {update_all_result.modified_count} dokumen menjadi status 'Tersedia'.")

# 2. Mengubah status menjadi 'Habis' khusus untuk 2 menu pilihan
collection.update_one({"nama_menu": "Nasi Goreng Rempah"}, {"$set": {"status": "Habis"}})
collection.update_one({"nama_menu": "Espresso Single"}, {"$set": {"status": "Habis"}})
print("update_one() berhasil mengubah status 'Nasi Goreng Rempah' dan 'Espresso Single' menjadi 'Habis'.\n")

update_many() berhasil memperbarui 12 dokumen menjadi status 'Tersedia'.
update_one() berhasil mengubah status 'Nasi Goreng Rempah' dan 'Espresso Single' menjadi 'Habis'.



In [8]:
# Mengambil dokumen dari MongoDB dan mengonversi ObjectId ke String agar kompatibel dengan Spark
raw_docs = list(collection.find({}))
for doc in raw_docs:
    doc["_id"] = str(doc["_id"])

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas9") \
    .master("local[*]") \
    .getOrCreate()

# Konversi List Dokumen BSON/Python ke Spark DataFrame
df_menu = spark.createDataFrame(raw_docs)

print("Skema Spark DataFrame (printSchema):")
df_menu.printSchema()

print("\nRata-rata Harga per Kategori:")
df_kategori_avg = df_menu.groupBy("kategori") \
    .agg(spark_round(avg("harga"), 2).alias("rata_rata_harga"))
df_kategori_avg.show()

26/11/04 23:46:29 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 192.168.100.6 instead (on interface wlp2s0)
26/11/04 23:46:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 23:46:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Skema Spark DataFrame (printSchema):
root
 |-- _id: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- status: string (nullable = true)
 |-- stok: long (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- manis_level: long (nullable = true)


Rata-rata Harga per Kategori:


+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|       21666.67|
| Makanan|        23400.0|
|Non-Kopi|        26000.0|
+--------+---------------+



### KESIMPULAN

Tantangan utama saat mengonversi data dari MongoDB ke Apache Spark DataFrame terletak pada perbedaan mendasar antara arsitektur **schemaless (fleksibel)** berbasis dokumen BSON dengan arsitektur **rigid schema (terstruktur)** berbasis tabular pada Spark. 

Pada MongoDB, setiap dokumen dalam collection `menu` memiliki field yang tidak seragam—misalnya atribut `tingkat_kepahitan` hanya ada pada kategori "Kopi", sedangkan `pedas_level` hanya dimiliki oleh "Makanan". Ketika data heterogen ini diimpor ke dalam Spark DataFrame, Spark harus melakukan konsolidasi skema secara eksplisit (*schema inference*). Tantangan yang muncul meliputi:
1. **Pencegahan Tipe Data Inkompatibel:** Objek native MongoDB seperti `ObjectId` pada field `_id` tidak dapat diserialisasi secara langsung oleh encoder Spark tanpa konversi ke bentuk `StringType`.
2. **Penanganan Field Kosong (*Missing Attributes*):** Spark perlu memetakan seluruh field unik yang ditemukan di seluruh dokumen menjadi kolom tabel. Dokumen yang tidak memiliki atribut tertentu akan secara otomatis diisi dengan nilai `null`.

**Solusi yang Diterapkan:**
Untuk mengatasi masalah ini, dilakukan pra-pemrosesan sederhana dengan mengonversi tipe data `ObjectId` menjadi `String` sebelum dokumen dimasukkan ke dalam `SparkSession.createDataFrame()`. Secara otomatis, Spark membentuk skema bersatu (*unified schema*) di mana field beropsional (seperti `pedas_level`, `tingkat_kepahitan`, dan `manis_level`) tetap teridentifikasi sebagai kolom terstruktur dengan nilai `null` bagi dokumen yang tidak menyatakannya. Hal ini memungkinkan operasi agregasi relasional—seperti `groupBy("kategori").agg(avg("harga"))`—berjalan dengan presisi tanpa terganggu oleh ketidakseragaman atribut antar dokumen.